# M2 encoder bake-off — free T4 GPU (Google Colab)

Runs the GYF M2 embedding bake-off (`ml/eval/bake_off.py`) on Colab's **free T4 GPU**: re-embeds the catalog with the incumbent + the SigLIP-2 research candidates and ranks them through the M1 regression gate. No paid GPU, no HF Pro.

**Before running:** `Runtime → Change runtime type → T4 GPU`, then `Runtime → Run all`.

Reproducible end-to-end: the catalog (gitignored locally) is regenerated from the public `ashraq/fashion-product-images-small` dataset, so nothing needs uploading. Embeddings compute locally on the T4 (the remote ZeroGPU lane is left unset).

## 1. Confirm a GPU is attached

In [ ]:
!nvidia-smi -L || print('No GPU — set Runtime > Change runtime type > T4 GPU')

## 2. Clone the private repo
Create a GitHub token with `repo` read scope (github.com → Settings → Developer settings → Personal access tokens). It is read via `getpass` and never stored in the notebook.

In [ ]:
import getpass, os, subprocess

token = getpass.getpass('GitHub token (repo read): ').strip()
REPO = 'GetYourFit/GYF-V2'
if not os.path.isdir('/content/GYF-V2'):
    url = f'https://{token}@github.com/{REPO}.git'
    subprocess.run(['git', 'clone', '--depth', '1', url, '/content/GYF-V2'], check=True)
os.chdir('/content/GYF-V2')
del token, url  # drop the credential from memory
print('cwd:', os.getcwd())

## 3. Install dependencies
The shared contracts + the `ml` package (perception extra: torch, open_clip) + `datasets` for regenerating the catalog.

In [ ]:
!pip install -q -e packages/contracts
!pip install -q -e 'ml[perception]'
!pip install -q datasets

## 4. Regenerate the catalog (public dataset, ~12 items/category)

In [ ]:
import os
os.environ['HF_HOME'] = '/content/hf-cache'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
# leave GYF_ENCODER_REMOTE_URL unset -> bake-off uses the local SiglipEncoder on the T4 GPU
!python scripts/seed_fashion_feed.py --per-category 12 --out data/e2e
!ls data/e2e && echo 'images:' && ls data/e2e/images | wc -l

## 5. Run the bake-off on the T4
Re-embeds the catalog with each registry `encoder` model and ranks candidates vs the incumbent (gate metric: MRR). First run downloads each model's weights.

In [ ]:
import os
os.environ['HF_HOME'] = '/content/hf-cache'
!cd ml && python -m eval.bake_off

## 6. Download the EvalReports
Commit these to `eval-reports/bakeoffs/` in the repo as the M2 evidence (milestone-done discipline).

In [ ]:
import shutil
from google.colab import files
shutil.make_archive('/content/m2-bakeoff-reports', 'zip', 'eval-reports/bakeoffs')
files.download('/content/m2-bakeoff-reports.zip')